In [ ]:
import csv
import os
import time
from datetime import date, datetime
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException, NoSuchElementException, ElementClickInterceptedException
import shutil
from google.colab import drive

# Montagem do Google Drive para salvar os dados
drive.mount('/content/drive')

# Definir o caminho para salvar o arquivo CSV na pasta "output" no Google Drive
output_dir = '/content/drive/MyDrive/output'
if not os.path.exists(output_dir):
    os.makedirs(output_dir)
output_path = os.path.join(output_dir, 'tumelero_produtos.csv')

# Configuração do WebDriver para uso no Colab
chrome_options = webdriver.ChromeOptions()
chrome_options.add_argument('--headless')
chrome_options.add_argument('--no-sandbox')
chrome_options.add_argument('--disable-dev-shm-usage')

# Inicializa o WebDriver
driver = webdriver.Chrome(options=chrome_options)

# Abre a página do Tumelero
driver.get('https://www.tumelero.com.br/pisos-e-revestimentos/pisos/pisos-ceramicos')

# Espera até que os elementos de produtos estejam visíveis
try:
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, 'li[data-smarthintfeature="search"]'))
    )
except TimeoutException:
    print("Tempo de espera expirou. Não foi possível encontrar os produtos.")
    driver.quit()
    exit()

# Função para fechar possíveis overlays
def fechar_overlay():
    try:
        overlays = driver.find_elements(By.CSS_SELECTOR, 'div.onetrust-pc-dark-filter')
        for overlay in overlays:
            driver.execute_script("arguments[0].click();", overlay)
            time.sleep(1)  # Aguarda a remoção do overlay
    except NoSuchElementException:
        print("Nenhum overlay encontrado.")

# Função para extrair dados da página
def extrair_dados():
    dados = []
    pagina_atual = 1

    while True:
        try:
            produtos = driver.find_elements(By.CSS_SELECTOR, 'li[data-smarthintfeature="search"]')
            print(f"Total de produtos encontrados nesta página: {len(produtos)}")

            for produto in produtos:
                try:
                    nome = produto.find_element(By.CSS_SELECTOR, 'h2.x-shelf__title').text
                    # Atualize o seletor conforme necessário
                    try:
                        preco = produto.find_element(By.CSS_SELECTOR, 'span.x-shelf__best-price strong').text
                    except NoSuchElementException:
                        preco = 'Preço não disponível'
                    imagem = produto.find_element(By.CSS_SELECTOR, 'div.x-shelf__img-container img').get_attribute('src')
                    link = produto.find_element(By.CSS_SELECTOR, 'div.x-shelf__img-container a').get_attribute('href')

                    dados.append({
                        'nome': nome,
                        'preco': preco,
                        'imagem': imagem,
                        'link': link
                    })
                except NoSuchElementException as e:
                    print(f"Elemento não encontrado dentro do produto: {e}")
                    continue

            try:
                # Tenta encontrar e fechar possíveis overlays
                fechar_overlay()

                # Tenta encontrar o botão "Mais produtos"
                botao_mais = WebDriverWait(driver, 20).until(
                    EC.presence_of_element_located((By.CSS_SELECTOR, 'div.father-loadmore-smarthint button.cf-load-more'))
                )
                driver.execute_script("arguments[0].scrollIntoView(true);", botao_mais)  # Scroll para o botão

                # Tenta clicar no botão e lida com overlays
                try:
                    WebDriverWait(driver, 20).until(
                        EC.element_to_be_clickable((By.CSS_SELECTOR, 'div.father-loadmore-smarthint button.cf-load-more'))
                    ).click()
                    time.sleep(2)  # Espera o carregamento dos produtos
                except ElementClickInterceptedException:
                    print("Overlay ou outro elemento interceptou o clique. Tentando novamente...")
                    driver.execute_script("arguments[0].click();", botao_mais)
                    time.sleep(2)

                pagina_atual += 1
                print(f"Next page: {pagina_atual}")

            except NoSuchElementException:
                print("Não foi possível encontrar o botão 'Mais produtos'.")
                break
            except StaleElementReferenceException:
                print("O botão 'Mais produtos' não está mais disponível. Encerrando coleta.")
                break

        except NoSuchElementException as e:
            print(f"Erro ao encontrar produtos: {e}")
            break

    return dados

# Execução principal
dados = extrair_dados()

# Verifica se dados foram coletados
if not dados:
    print("Nenhum dado coletado.")
else:
    # Exibe os dados coletados no console
    for dado in dados:
        print(f"Nome: {dado['nome']}, Preço: {dado['preco']}, Imagem: {dado['imagem']}, Link: {dado['link']}")

# Salva os dados no arquivo CSV
if dados:
    with open(output_path, 'w', newline='', encoding='utf-8') as file:
        writer = csv.DictWriter(file, fieldnames=['nome', 'preco', 'imagem', 'link'])
        writer.writeheader()
        writer.writerows(dados)
    print(f"Dados salvos em {output_path}")

# Fechando o navegador
driver.quit()


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total de produtos encontrados nesta página: 36
Overlay ou outro elemento interceptou o clique. Tentando novamente...
Next page: 2
Total de produtos encontrados nesta página: 72
Overlay ou outro elemento interceptou o clique. Tentando novamente...
Next page: 3
Total de produtos encontrados nesta página: 108
Overlay ou outro elemento interceptou o clique. Tentando novamente...
Next page: 4
Total de produtos encontrados nesta página: 144
Overlay ou outro elemento interceptou o clique. Tentando novamente...
Next page: 5
Total de produtos encontrados nesta página: 180
Overlay ou outro elemento interceptou o clique. Tentando novamente...
Next page: 6
Total de produtos encontrados nesta página: 216
Overlay ou outro elemento interceptou o clique. Tentando novamente...
Next page: 7
Total de produtos encontrados nesta página: 225


TimeoutException: Message: 
Stacktrace:
#0 0x58437a63981a <unknown>
#1 0x58437a307e50 <unknown>
#2 0x58437a357644 <unknown>
#3 0x58437a357931 <unknown>
#4 0x58437a39d534 <unknown>
#5 0x58437a37c4bd <unknown>
#6 0x58437a39a9c6 <unknown>
#7 0x58437a37c233 <unknown>
#8 0x58437a34a093 <unknown>
#9 0x58437a34b09e <unknown>
#10 0x58437a600a7b <unknown>
#11 0x58437a604a31 <unknown>
#12 0x58437a5ec645 <unknown>
#13 0x58437a6055a2 <unknown>
#14 0x58437a5d181f <unknown>
#15 0x58437a628618 <unknown>
#16 0x58437a6287e2 <unknown>
#17 0x58437a63860c <unknown>
#18 0x7b539f5beac3 <unknown>
